# Lista encadeada: busca, inserção e remoção — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

A aula passada terminou com uma lista que só crescia pela frente. Este tutorial retoma o programa
`lista.c` visto hoje — uma lista mantida em ordem crescente, com `busca`, `insere_ordenado` e
`remove_valor` — e segue o mesmo caminho dos slides: buscar, inserir em qualquer posição, remover,
o ponteiro para ponteiro e o custo de cada operação. Depois de cada bloco há um exercício com
teste automático; no fim vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Buscar um valor numa lista encadeada, protegendo todo acesso com `p != NULL` na ordem certa
- Inserir um nó em qualquer posição, parando no nó anterior e ligando os ponteiros na ordem certa
- Remover um nó tratando os três casos: não achou, achou o primeiro, achou outro
- Usar um ponteiro para ponteiro (`No **`) para escrever inserção e remoção sem caso especial
- Estimar quantos nós cada operação visita e medir o efeito disso no tempo

In [ ]:
# Confira se o gcc está disponível e defina a função que testa os exercícios
!gcc --version | head -1

import subprocess

def testa(comando, esperado, entrada=None, tempo=20):
    '''Roda o comando e compara a saída com o esperado.'''
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True,
                           input=entrada, timeout=tempo)
    except subprocess.TimeoutExpired:
        print(f'Demorou mais de {tempo} s: a função está percorrendo a lista à toa?')
        return
    saida = r.stdout.strip()
    if r.returncode != 0:
        print(f'O programa terminou com erro (status {r.returncode}; '
              '-11 ou 139 = segmentation fault).')
        if r.stderr:
            print(r.stderr[:1500])
    if saida == esperado.strip():
        print('OK!')
    else:
        print('Saída esperada:')
        print(esperado.strip())
        print('Saída obtida:')
        print(saida)

## O programa da aula

Recompile o programa condutor **exatamente como saiu da aula**. Antes de executar, escreva a saída
que você espera: em que ordem os valores aparecem, o que as buscas respondem e como fica a lista
depois das três remoções.

In [ ]:
%%writefile lista.c
/* lista.c --- lista em ordem crescente: busca, insercao e remocao */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

/* devolve o primeiro no com valor x, ou NULL */
No *busca(No *ini, int x)
{
    No *p = ini;
    while (p != NULL && p->valor != x)
        p = p->prox;
    return p;
}

/* insere x mantendo a ordem; devolve o novo inicio */
No *insere_ordenado(No *ini, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    if (ini == NULL || x < ini->valor) {   /* vira o primeiro */
        novo->prox = ini;
        return novo;
    }
    No *ant = ini;
    while (ant->prox != NULL && ant->prox->valor < x)
        ant = ant->prox;
    novo->prox = ant->prox;
    ant->prox = novo;
    return ini;
}

/* remove o primeiro no com valor x; devolve o novo inicio */
No *remove_valor(No *ini, int x)
{
    No *ant = NULL, *p = ini;
    while (p != NULL && p->valor != x) {
        ant = p;
        p = p->prox;
    }
    if (p == NULL)          /* nao achou */
        return ini;
    if (ant == NULL)        /* era o primeiro */
        ini = p->prox;
    else
        ant->prox = p->prox;
    free(p);
    return ini;
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

int main(void)
{
    int entrada[] = {30, 10, 40, 20};
    No *lista = NULL;
    for (int i = 0; i < 4; i++)
        lista = insere_ordenado(lista, entrada[i]);
    imprime(lista);
    printf("busca 20: %s\n", busca(lista, 20) ? "achou" : "nao achou");
    printf("busca 25: %s\n", busca(lista, 25) ? "achou" : "nao achou");
    lista = remove_valor(lista, 10);
    lista = remove_valor(lista, 40);
    lista = remove_valor(lista, 99);
    imprime(lista);
    libera(lista);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra lista.c -o lista && ./lista

## 1. Buscar

Uma lista não tem índice: a única forma de achar um valor é começar no início e seguir os `prox`,
comparando cada nó, até **achar** (devolve o nó) ou chegar ao **`NULL`** (devolve `NULL`). A
condição do laço é `p != NULL && p->valor != x` — e a ordem importa: o `&&` avalia da esquerda
para a direita e **para no primeiro falso**. Com os testes trocados, `p->valor` é lido quando `p`
já é `NULL`.

O programa abaixo é a `busca` com a condição trocada. Compile sem e com `-O2` e compare.

In [ ]:
!sed '/^No \*busca/,/^}/ s/while (p != NULL \&\& p->valor != x)/while (p->valor != x \&\& p != NULL)/' lista.c > busca_errada.c
!grep -n "while (p->valor" busca_errada.c
!gcc -Wall -Wextra busca_errada.c -o busca_errada && ./busca_errada; echo "status: $?"
!gcc -O2 -Wall -Wextra busca_errada.c -o busca_errada-O2 && ./busca_errada-O2; echo "status: $?"

Sem `-O2`, o programa é derrubado (status `139` = 128 + 11, o sinal `SIGSEGV`). Aqui no Jupyter
talvez **nenhuma** linha apareça antes da queda: quando a saída vai para um *pipe*, o `printf`
guarda o texto num buffer e só o entrega no fim do programa — e um programa derrubado não chega ao
fim. No terminal, as duas primeiras linhas aparecem.

Com `-O2`, o programa costuma não cair — e responder que **achou o 25**. Ler `p->valor` com
`p == NULL` é comportamento indefinido; o compilador pode supor que isso nunca acontece, concluir
que `p` nunca é `NULL` e descartar o “não achou”. Um erro de memória não precisa derrubar o
programa: ele pode só produzir uma resposta errada.

### Exercício 1 — busca que desiste cedo

Numa lista **em ordem crescente**, a busca pode parar assim que encontrar um valor maior que `x`:
dali para a frente, todos são maiores. Escreva `busca_ordenada`, que também conta em `*visitados`
quantos nós foram comparados com `x`.

In [ ]:
%%writefile ex1.c
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

/* --- funcoes da aula (nao precisa mexer) --- */
No *insere_inicio(No *ini, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) { perror("malloc"); exit(1); }
    novo->valor = x;
    novo->prox = ini;
    return novo;
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

/* Exercicio 1: a lista esta em ordem crescente. Devolva o no com valor x,
   ou NULL se ele nao estiver na lista, parando assim que passar de x.
   Em *visitados, guarde quantos nos foram comparados com x.
   TODO: implemente busca_ordenada. */
No *busca_ordenada(No *ini, int x, int *visitados)
{
    /* TODO: implemente aqui */
    *visitados = 0;
    return NULL;
}

int main(void)
{
    No *lista = NULL;
    for (int v = 40; v >= 10; v -= 10)
        lista = insere_inicio(lista, v);            /* 10 20 30 40 */
    int xs[] = {25, 30, 5, 50};
    for (int i = 0; i < 4; i++) {
        int vis = -1;
        No *r = busca_ordenada(lista, xs[i], &vis);
        printf("%d:%c%d ", xs[i], r != NULL && r->valor == xs[i] ? 'S' : 'N', vis);
    }
    int vis = -1;
    printf("vazia:%c%d\n", busca_ordenada(NULL, 7, &vis) ? 'S' : 'N', vis);
    libera(lista);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra ex1.c -o ex1
testa('./ex1', '25:N3 30:S3 5:N1 50:N4 vazia:N0')

Para `x = 50`, a busca ordenada visita tantos nós quanto a busca comum. Em que casos ela ganha?
Em média, ela olha metade da lista — o número de passos ainda cresce com $n$.

## 2. Inserir em qualquer posição

A lista só tem setas para a frente: para encaixar um nó, é preciso parar no nó **anterior** à
posição, porque é o `prox` dele que vai mudar. Os dois passos são sempre nesta ordem:

```c
novo->prox = ant->prox;   /* 1. o novo aponta para quem vinha depois */
ant->prox = novo;         /* 2. o anterior passa a apontar para o novo */
```

O fim da lista não é especial (o passo 1 copia o `NULL`). O início é: o primeiro nó não tem
anterior, e quem aponta para ele é a variável do `main`.

Inserir no fim **percorrendo** a lista custa $n$ passos por inserção. O programa abaixo mede isso.

In [ ]:
%%writefile fim_vs_inicio.c
/* fim_vs_inicio.c --- inserir n valores no fim (percorrendo) e no inicio */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

static double agora(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + t.tv_nsec / 1e9;
}

static No *novo_no(int x, No *prox)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->prox = prox;
    return novo;
}

No *insere_inicio(No *ini, int x)
{
    return novo_no(x, ini);
}

/* percorre ate o ultimo a cada insercao */
No *insere_fim(No *ini, int x)
{
    No *novo = novo_no(x, NULL);
    if (ini == NULL)
        return novo;
    No *p = ini;
    while (p->prox != NULL)
        p = p->prox;
    p->prox = novo;
    return ini;
}

static void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 20000;

    double t0 = agora();
    No *a = NULL;
    for (int i = 0; i < n; i++)
        a = insere_inicio(a, i);
    double t_inicio = agora() - t0;

    t0 = agora();
    No *b = NULL;
    for (int i = 0; i < n; i++)
        b = insere_fim(b, i);
    double t_fim = agora() - t0;

    printf("n = %d: inicio %.4f s, fim %.4f s\n", n, t_inicio, t_fim);
    libera(a);
    libera(b);
    return 0;
}

In [ ]:
# Visualização: inserir n valores no início e no fim (percorrendo a lista)
import matplotlib.pyplot as plt

!gcc -O2 -Wall -Wextra fim_vs_inicio.c -o fim_vs_inicio

ns, t_ini, t_fim = [], [], []
for n in [5000, 10000, 20000, 30000, 40000, 60000]:
    s = subprocess.run(['./fim_vs_inicio', str(n)], capture_output=True, text=True).stdout
    partes = s.replace(',', '').split()          # n = N: inicio T s fim T s
    ns.append(n)
    t_ini.append(float(partes[4]))
    t_fim.append(float(partes[7]))
    print(s.strip())

plt.plot(ns, t_fim, 'o-', label='no fim (percorrendo)')
plt.plot(ns, t_ini, 's-', label='no início')
plt.xlabel('n (valores inseridos)')
plt.ylabel('tempo (s)')
plt.title('Inserir n valores numa lista encadeada')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

A curva do fim é uma parábola: dobrar $n$ quadruplica o tempo, porque são
$0 + 1 + \cdots + (n-1) \approx n^2/2$ passos no total.

### Exercício 2 — inserir no fim sem percorrer

Guarde, junto com o início, um ponteiro para o **último** nó. Assim `insere_fim` não precisa
procurá-lo. Pense: o que muda quando a lista está vazia?

In [ ]:
%%writefile ex2.c
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

typedef struct {
    No *ini;   /* primeiro no, ou NULL */
    No *fim;   /* ultimo no, ou NULL */
} Lista;

/* --- funcoes da aula (nao precisa mexer) --- */
No *insere_inicio(No *ini, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) { perror("malloc"); exit(1); }
    novo->valor = x;
    novo->prox = ini;
    return novo;
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

/* Exercicio 2: insere x no fim da lista l, SEM percorrer a lista.
   Mantenha l->ini e l->fim corretos, inclusive quando a lista esta vazia.
   TODO: implemente insere_fim. Ela deve inserir 300 mil valores em menos
   de um segundo. */
void insere_fim(Lista *l, int x)
{
    /* TODO: implemente aqui */
}

int main(void)
{
    Lista l = {NULL, NULL};
    for (int i = 1; i <= 5; i++)
        insere_fim(&l, i);
    imprime(l.ini);
    for (int i = 1; i <= 300000; i++)
        insere_fim(&l, i);
    int n = 0;
    for (const No *p = l.ini; p != NULL; p = p->prox)
        n++;
    printf("%d %d\n", n, l.fim->valor);
    libera(l.ini);
    return 0;
}

In [ ]:
!gcc -O2 -Wall -Wextra ex2.c -o ex2
testa('./ex2', '''1 -> 2 -> 3 -> 4 -> 5 -> NULL
300005 300000''', tempo=5)

## 3. Remover

Remover é desviar a seta do anterior e **só então** liberar o nó:

```c
ant->prox = p->prox;
free(p);
```

Por isso a busca anda com dois ponteiros, `ant` um passo atrás de `p`. São três casos: não achou
(`p == NULL`, nada a fazer), achou o primeiro (`ant == NULL`: muda o início) e achou outro. Sem o
caso do primeiro, `ant->prox` escreve no endereço `NULL + 8` — o campo `prox` fica 8 bytes depois
do início do nó.

### Exercício 3 — remover por posição

Escreva `remove_pos`, que remove o nó da posição `k` (contando de 0) e devolve o novo início. Se a
posição não existir — `k` negativo ou maior que o último índice —, a lista fica como está. O
teste compila com `-fsanitize=address`: um `free` na hora errada aparece na saída.

In [ ]:
%%writefile ex3.c
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

/* --- funcoes da aula (nao precisa mexer) --- */
No *insere_inicio(No *ini, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) { perror("malloc"); exit(1); }
    novo->valor = x;
    novo->prox = ini;
    return novo;
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

/* Exercicio 3: remove o no da posicao k (a partir de 0), se ela existir,
   e devolve o novo inicio da lista.
   TODO: implemente remove_pos. Cuidado com k == 0. */
No *remove_pos(No *ini, int k)
{
    /* TODO: implemente aqui */
    return ini;
}

int main(void)
{
    No *lista = NULL;
    for (int v = 50; v >= 10; v -= 10)
        lista = insere_inicio(lista, v);            /* 10 20 30 40 50 */
    lista = remove_pos(lista, 0);                   /* tira o 10 */
    lista = remove_pos(lista, 3);                   /* tira o 50 */
    lista = remove_pos(lista, 7);                   /* nao existe */
    lista = remove_pos(lista, -1);                  /* nao existe */
    lista = remove_pos(lista, 1);                   /* tira o 30 */
    imprime(lista);
    imprime(remove_pos(NULL, 0));
    libera(lista);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex3.c -o ex3
testa('./ex3', '''20 -> 40 -> NULL
NULL''')

## 4. Ponteiro para ponteiro

O caso especial do primeiro nó existe porque a seta que precisa mudar às vezes é a variável
`lista` do `main` e às vezes o `prox` de um nó. As duas são do tipo `No *` — e um `No **` pode
apontar para qualquer uma delas. Em vez de guardar o nó anterior, a busca guarda **o endereço da
seta que chega ao nó atual**:

- começa em `pp = &lista` (passado pelo `main`);
- avança com `pp = &(*pp)->prox` — o endereço do campo `prox` do nó atual;
- muda a seta com `*pp = ...`, seja ela `lista` ou um `prox`.

Os parênteses em `(*pp)->valor` são obrigatórios: `->` tem precedência maior que `*`.

In [ ]:
%%writefile ponteiro_duplo.c
/* ponteiro_duplo.c --- insercao e remocao sem caso especial */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

/* pp aponta para o ponteiro que sera mudado: o inicio ou um prox */
void insere_ordenado(No **pp, int x)
{
    while (*pp != NULL && (*pp)->valor < x)
        pp = &(*pp)->prox;
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->prox = *pp;
    *pp = novo;
}

void remove_valor(No **pp, int x)
{
    while (*pp != NULL && (*pp)->valor != x)
        pp = &(*pp)->prox;
    if (*pp != NULL) {
        No *p = *pp;
        *pp = p->prox;
        free(p);
    }
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

int main(void)
{
    int entrada[] = {30, 10, 40, 20};
    No *lista = NULL;
    for (int i = 0; i < 4; i++)
        insere_ordenado(&lista, entrada[i]);
    imprime(lista);
    remove_valor(&lista, 10);
    remove_valor(&lista, 40);
    remove_valor(&lista, 99);
    imprime(lista);
    libera(lista);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra ponteiro_duplo.c -o ponteiro_duplo && ./ponteiro_duplo

### Exercício 4 — remover todas as ocorrências

Usando `No **`, escreva `remove_todos`, que tira **todas** as ocorrências de `x`. Atenção: depois
de remover um nó, `*pp` já aponta para o seguinte — não avance `pp` nesse caso.

In [ ]:
%%writefile ex4.c
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

/* --- funcoes da aula (nao precisa mexer) --- */
No *insere_inicio(No *ini, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) { perror("malloc"); exit(1); }
    novo->valor = x;
    novo->prox = ini;
    return novo;
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

/* Exercicio 4: remove todos os nos com valor x.
   pp aponta para a seta que chega ao no atual (no inicio, &lista).
   TODO: implemente remove_todos sem nenhum caso especial para o
   primeiro no. */
void remove_todos(No **pp, int x)
{
    /* TODO: implemente aqui */
}

int main(void)
{
    int v[] = {10, 30, 10, 20, 10, 10};
    No *lista = NULL;
    for (int i = 0; i < 6; i++)
        lista = insere_inicio(lista, v[i]);         /* 10 10 20 10 30 10 */
    remove_todos(&lista, 10);
    imprime(lista);
    remove_todos(&lista, 99);
    imprime(lista);
    remove_todos(&lista, 30);
    remove_todos(&lista, 20);
    imprime(lista);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex4.c -o ex4
testa('./ex4', '''20 -> 30 -> NULL
20 -> 30 -> NULL
NULL''')

## Tarefas: altere o programa da aula

Volte à célula de `lista.c`, faça cada alteração, reexecute e confira se a saída bate com o que
você previu.

1. **Busca que desiste cedo.** Troque a `busca` de `lista.c` pela sua `busca_ordenada` do
   Exercício 1. Quantos nós ela visita em `busca(lista, 25)`? E a versão original?
2. **Valores repetidos.** Insira o 20 uma segunda vez. O novo 20 entra antes ou depois do antigo?
   Troque `<` por `<=` no laço de `insere_ordenado` e preveja o que muda antes de executar.
   (Dica: imprima também o endereço de cada nó, com `%p`.)
3. **Remover sem caso especial.** Reescreva `remove_valor` de `lista.c` com `No **`, como em
   `ponteiro_duplo.c`, e ajuste o `main`. Quantas linhas a função perdeu?

## Desafio — uma agenda sempre em ordem alfabética

Complete o programa abaixo. Ele lê comandos da entrada padrão, um por linha, até o fim do arquivo,
e mantém os nomes numa lista encadeada **em ordem alfabética** (use `strcmp`):

| comando | efeito |
|---|---|
| `+ nome` | insere o nome em ordem, se ainda não estiver lá |
| `- nome` | remove o nome, se estiver lá |
| `? nome` | imprime `sim` ou `nao` |
| `=` | imprime todos os nomes numa linha, separados por espaço |

Cada nome mora num bloco do `malloc` com o tamanho exato — e precisa ser liberado junto com o nó.
Use `Contato **` para não ter caso especial. Com `-fsanitize=address`, nenhum erro.

In [ ]:
%%writefile agenda.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct contato {
    char *nome;
    struct contato *prox;
} Contato;

/* TODO: insere uma COPIA de nome (malloc do tamanho exato + strcpy), em
   ordem alfabetica, se ele ainda nao estiver na agenda. */
void insere(Contato **pp, const char *nome)
{
}

/* TODO: remove o contato com esse nome, se existir (dois free: quais?) */
void remove_nome(Contato **pp, const char *nome)
{
}

/* TODO: devolve 1 se o nome esta na agenda, 0 caso contrario */
int contem(const Contato *ini, const char *nome)
{
    return 0;
}

/* TODO: imprime os nomes numa linha, separados por espaco */
void imprime(const Contato *ini)
{
    printf("\n");
}

/* TODO: libera todos os contatos e todos os nomes */
void libera(Contato *ini)
{
}

int main(void)
{
    Contato *agenda = NULL;
    char linha[128];
    while (fgets(linha, sizeof linha, stdin) != NULL) {
        linha[strcspn(linha, "\n")] = '\0';
        char cmd = linha[0];
        const char *nome = (linha[0] != '\0' && linha[1] == ' ') ? linha + 2 : "";
        if (cmd == '+')
            insere(&agenda, nome);
        else if (cmd == '-')
            remove_nome(&agenda, nome);
        else if (cmd == '?')
            printf("%s\n", contem(agenda, nome) ? "sim" : "nao");
        else if (cmd == '=')
            imprime(agenda);
    }
    libera(agenda);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra agenda.c -o agenda
entrada = '''+ Maria
+ Ana
+ Joao
+ Ana
? Ana
? Pedro
=
- Ana
- Pedro
+ Beatriz
=
- Maria
- Joao
- Beatriz
=
'''
testa('./agenda', '''sim
nao
Ana Joao Maria
Beatriz Joao Maria''', entrada=entrada)

Repare que a última linha esperada é vazia: depois de remover todos, `=` imprime só a quebra de
linha. Teste também com nomes que diferem só em maiúsculas (`ana` e `Ana`): em que ordem o
`strcmp` os coloca, e por quê?

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Os pontos principais desta aula estão
em:

- Celes, Cerqueira e Rangel (2004), *Introdução a estruturas de dados*, cap. 10 — busca, remoção e
  lista ordenada;
- Feofiloff (2009), *Algoritmos em linguagem C*, cap. 4 — listas encadeadas;
- King (2008), *C Programming: A Modern Approach*, §17.5 (listas encadeadas) e §17.6 (ponteiros
  para ponteiros);
- Cormen et al. (2009), *Introduction to Algorithms*, §10.2 — `LIST-SEARCH`, `LIST-INSERT`,
  `LIST-DELETE`;
- ISO/IEC 9899:2018, §6.5.13 — o operador `&&` e a avaliação em curto-circuito.